# The Short-Time Fourier Transform

Window a tone to tame leakage, follow a chirp through time, and check the NumPy STFT against librosa.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
from main import energy_outside_peak, hann_window, istft, linear_chirp, resolution, sine_wave, stft, stft_frequencies

tone = sine_wave(442.0, 8_000, 0.25, amplitude=0.5)
rectangular = np.abs(np.fft.rfft(tone))
windowed = np.abs(np.fft.rfft(tone * hann_window(len(tone))))
round(energy_outside_peak(rectangular), 4), round(energy_outside_peak(windowed), 6)

In [ ]:
sample_rate, n_fft, hop_length = 16_000, 512, 128
chirp = linear_chirp(300.0, 3_000.0, sample_rate, 1.0)
spectrum = stft(chirp, n_fft, hop_length)
peaks = stft_frequencies(sample_rate, n_fft)[np.argmax(np.abs(spectrum), axis=0)]
print(resolution(sample_rate, n_fft, hop_length), spectrum.shape)
peaks[::25]

Each column is one frame, so the peak frequency climbs with the chirp. Change `n_fft` to 128 and to 2048. Which setting follows the sweep smoothly, and which one gives the sharpest peak in each column?

In [ ]:
import librosa

reference = librosa.stft(chirp, n_fft=n_fft, hop_length=hop_length)
restored = istft(spectrum, hop_length, length=len(chirp))
print(f'NumPy vs librosa.stft: max |difference| = {np.max(np.abs(spectrum - reference)):.2e}')
print(f'STFT -> ISTFT round trip: max |difference| = {np.max(np.abs(restored - chirp)):.2e}')

librosa uses the same defaults as `code/main.py`: a periodic Hann window, `center=True`, and zero padding by `n_fft // 2`. Try `librosa.stft(chirp, n_fft=n_fft, hop_length=hop_length, center=False)` and explain why the frame count changes.